# 复合层

到目前为止，我们的网络一直只有一层：一个线性层直接把输入映射到输出。这类单层线性模型只能拟合线性关系，面对更复杂的数据规律就无能为力了。

真正的深度学习依赖**多层网络**：数据经过一层又一层的变换，每一层学习上一层输出的，更抽象的数据，最终提炼出预测结果。

本章通过引入**复合层**（Composite Layer）来实现多层支持。复合层的核心思想很简单：它是一个**层的容器**，对外提供和单层完全相同的接口（`forward`、`parameters`），内部则将多个子层按某种逻辑串联起来。这样，整个框架无需改动——模型类依然只看到一个层对象。

本章实现的第一个复合层是**顺序层**（Sequential）：把子层依次首尾相连，前一层的输出就是后一层的输入。

In [1]:
from abc import ABC, abstractmethod

import numpy as np

np.random.seed(99)

## 基础架构

### 张量

In [2]:
class Tensor:

    def __init__(self, data):
        self.data = np.array(data)
        self.grad = np.zeros_like(self.data)
        self.gradient_fn = None
        self.parents = set()

    def backward(self):
        if self.gradient_fn is not None:
            self.gradient_fn()

        for parent in self.parents:
            parent.backward()

    @property
    def shape(self):
        return self.data.shape

    def __repr__(self):
        return f'Tensor({self.data})'

### 基础数据集

In [3]:
class Dataset(ABC):

    def __init__(self, batch_size=1):
        self.batch_size = batch_size

        self.test_features = None
        self.test_labels = None
        self.train_features = None
        self.train_labels = None

        self.load()

        self.features = self.train_features
        self.labels = self.train_labels

    @abstractmethod
    def load(self):
        pass

    def train(self):
        self.features = self.train_features
        self.labels = self.train_labels

    def eval(self):
        self.features = self.test_features
        self.labels = self.test_labels

    def items(self):
        return Tensor(self.features), Tensor(self.labels)

    def __len__(self):
        return len(self.features) // self.batch_size

    def __getitem__(self, index):
        start = index * self.batch_size
        end = start + self.batch_size

        feature = Tensor(self.features[start: end])
        label = Tensor(self.labels[start: end])
        return feature, label

### 基础层

In [4]:
class Layer(ABC):

    def __call__(self, x: Tensor):
        return self.forward(x)

    @abstractmethod
    def forward(self, x: Tensor):
        pass

    @property
    def parameters(self):
        return []

    def __repr__(self):
        return f"{type(self).__name__}[]"

### 基础损失函数

In [5]:
class Loss(ABC):

    def __call__(self, p: Tensor, y: Tensor):
        return self.loss(p, y)

    @abstractmethod
    def loss(self, p: Tensor, y: Tensor):
        pass

### 基础优化器

In [6]:
class Optimizer(ABC):

    def __init__(self, parameters, lr):
        self.parameters = parameters
        self.lr = lr

    def reset(self):
        for p in self.parameters:
            p.grad = np.zeros_like(p.data)

    @abstractmethod
    def step(self):
        pass

### 基础模型

In [7]:
class Model(ABC):

    def __init__(self, layer, loss_fn, optimizer):
        self.layer = layer
        self.loss_fn = loss_fn
        self.optimizer = optimizer

    @abstractmethod
    def train(self, dataset, epochs):
        pass

    @abstractmethod
    def test(self, dataset):
        pass

## 数据

### 冰激凌销量数据集

In [8]:
class IcecreamDataset(Dataset):

    def load(self):
        self.train_features = [[22.5, 72.0],
                               [31.4, 45.0],
                               [19.8, 85.0],
                               [27.6, 63.0]]
        self.train_labels = [[95],
                             [210],
                             [70],
                             [155]]

        self.test_features = [[28.1, 58.0]]
        self.test_labels = [[165]]

## 模型

### 线性层

本章线性层的初始化方式发生了一个重要变化：**权重改为随机初始化**。

在单层网络中，所有权重设为相同值（如 `1/in_size`）也能正常训练。但在多层网络中，如果每层所有神经元的初始权重完全相同，它们在前向传播时会产生完全相同的输出，反向传播时产生完全相同的梯度，参数更新也完全相同。多个神经元永远保持对称，相当于只有一个神经元在工作，隐藏层的容量被完全浪费。这个问题称为**对称性问题**（Symmetry Problem）。

随机初始化打破了这种对称性，让每个神经元从不同起点出发，学习到数据的不同特征，这是多层网络能够工作的前提。

更加重要的是：我们在梯度计算函数 `gradient_fn` 里添加了计算输入值梯度 `x.grad` 的代码（参考第一部分中的梯度反向函数 `gradient_backward()`），并将输入值 `x` 加入父节点集合 `parents` 中。这样做的目的是**扩展计算图**，保证梯度可以在多个层之间实现反向传播。

In [9]:
class Linear(Layer):

    def __init__(self, in_size, out_size):
        self.weight = Tensor(np.random.rand(out_size, in_size) / in_size)
        self.bias = Tensor(np.random.rand(out_size))

    def forward(self, x: Tensor):
        p = Tensor(x.data @ self.weight.data.T + self.bias.data)

        def gradient_fn():
            self.weight.grad += p.grad.T @ x.data
            self.bias.grad += np.sum(p.grad, axis=0)
            x.grad += p.grad @ self.weight.data

        p.gradient_fn = gradient_fn
        p.parents = {x}
        return p

    @property
    def parameters(self):
        return [self.weight, self.bias]

    def __repr__(self):
        return f'Linear[weight{self.weight.shape}; bias{self.bias.shape}]'

### 顺序层

**顺序层**（Sequential）是第一个复合层：将子层列表按顺序串联，前一层的输出直接作为后一层的输入。

```{figure} images/sequential-layer.png
:align: center
:width: 400px
**图例：（两个子层）顺序层结构**

* $x_1, x_2$：输入特征值；
* $H$：第一个子层（隐藏层）的神经元；
* $O$：第二个子层（输出层）的神经元；
* $p$：输出预测值。

```

顺序层本身**没有任何可训练参数**，也不需要自己的梯度计算函数 `gradient_fn` 和父节点集合 `parents`。它只是一个容器，梯度链路由各子层的父节点集合 `parents` 自动连接。

顺序层唯一需要做的额外工作，是把所有子层的参数列表 `parameters` 汇总起来，交给优化器统一管理。

In [10]:
class Sequential(Layer):

    def __init__(self, layers):
        self.layers = layers

    def forward(self, x: Tensor):
        for l in self.layers:
            x = l(x)
        return x

    @property
    def parameters(self):
        return [p for l in self.layers for p in l.parameters]

    def __repr__(self):
        return '\n'.join(str(l) for l in self.layers if str(l))

### 均方误差损失函数

In [11]:
class MSELoss(Loss):

    def loss(self, p: Tensor, y: Tensor):
        mse = Tensor(np.mean(np.square(y.data - p.data)))

        def gradient_fn():
            p.grad += -2 * (y.data - p.data) / len(y.data)

        mse.gradient_fn = gradient_fn
        mse.parents = {p}
        return mse

### 随机梯度下降优化器

In [12]:
class SGDOptimizer(Optimizer):

    def step(self):
        for param in self.parameters:
            param.data -= param.grad * self.lr

### 神经网络模型

In [13]:
class NNModel(Model):

    def train(self, dataset, epochs):
        dataset.train()

        for epoch in range(epochs):
            for i in range(len(dataset)):
                features, labels = dataset[i]

                self.optimizer.reset()
                predictions = self.layer(features)
                loss = self.loss_fn(predictions, labels)
                loss.backward()
                self.optimizer.step()

    def test(self, dataset):
        dataset.eval()

        features, labels = dataset.items()
        predictions = self.layer(features)
        loss = self.loss_fn(predictions, labels)
        return predictions, loss

## 训练

### 超参数：学习率

In [14]:
LEARNING_RATE = 0.00001

### 超参数：批大小

In [15]:
BATCH_SIZE = 2

### 超参数：轮次

In [16]:
EPOCHS = 1000

### 建模

通过顺序层 `Sequential` 可以直接构建多层结构，无需修改模型类：

* 第一个线性层 `Linear(2, 4)` 是隐藏层，把 2 个输入特征映射到 4 个中间特征；
* 第二个线性层 `Linear(4, 1)` 是输出层，把 4 个中间特征映射到 1 个预测值。

层与层之间需要保证维度衔接：上一层的输出特征数 = 下一层的输入特征数。这里隐藏层输出 4，所以输出层的输入也是 4。

In [17]:
dataset = IcecreamDataset(BATCH_SIZE)

layer = Sequential([
    # 隐藏层：2 输入特征 → 4 中间特征
    Linear(2, 4),
    # 输出层：4 中间特征 → 1 预测值
    Linear(4, 1),
])
loss_fn = MSELoss()
optimizer = SGDOptimizer(layer.parameters, lr=LEARNING_RATE)

model = NNModel(layer, loss_fn, optimizer)
print(model.layer)

Linear[weight(4, 2); bias(4,)]
Linear[weight(1, 4); bias(1,)]


### 迭代

In [18]:
model.train(dataset, EPOCHS)

## 验证

### 测试

In [19]:
predictions, loss = model.test(dataset)
print(f'prediction: {predictions}')
print(f'loss:       {loss}')

prediction: Tensor([[166.58568956]])
loss:       Tensor(2.5144113737204954)


多层网络的预测约为 `166.6`，测试损失约 `2.51`。

## 课后练习

增加第二个隐藏层（三层网络），观察训练结果的变化。